# CSE475 - Phase 2 cross-dataset EVAL (Kaggle T4)

Loads the trained `phase2_dermnet_best.pt` (DermNet 23-class) and evaluates it on **5 test sets**:

1. `dermnet_test` — in-domain DermNet test split (23 classes, upper bound)
2. `skindiseasebd` — cross-dataset domain shift, 5 classes (paper headline)
3. `fitzpatrick_black` — dark-skin Fitzpatrick17k subset
4. `skin_disease_images` — phase-1 acne/rosacea holdout
5. `ddi_bias` — predicted-class distribution Black vs White (bias probe)

Test-set zips come from HF `Nirob-jon/cse475-eval-sets`; checkpoint from HF `Nirob-jon/cse475-skin-checkpoints`.

**Setup required:** add your HF token as a notebook **Secret** named `HF_TOKEN`.

In [ ]:
import os, getpass, zipfile, pathlib, shutil
from huggingface_hub import login, snapshot_download

CKPT_REPO = "Nirob-jon/cse475-skin-checkpoints"
EVAL_REPO = "Nirob-jon/cse475-eval-sets"
HARNESS = "multi-source-skin-disease-fusion"

token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
if not token:
    token = getpass.getpass("Paste your HF token (hf_...): ")
assert token, "No HF_TOKEN found"
login(token=token, add_to_git_credential=False)
print("logged in OK")

In [ ]:
# clone the harness (public repo)
!git clone --depth 1 https://github.com/nirjon001/multi-source-skin-disease-fusion.git
%cd multi-source-skin-disease-fusion

In [ ]:
# eval test sets (small; ~470 MB total) + DermNet split (1.7 GB, gives the in-domain test split + real train folder for class order)
EVAL_DIRS = {
    "skindiseasebd.zip": "skindiseasebd",
    "fitzpatrick_black.zip": "fitzpatrick_black",
    "skin_disease_images.zip": "skin_disease_images",
    "ddi.zip": "ddi",
}
DATA = pathlib.Path("/kaggle/working/data")
DATA.mkdir(parents=True, exist_ok=True)
zips = snapshot_download(EVAL_REPO, repo_type="dataset", allow_patterns="*.zip", local_dir=pathlib.Path("/kaggle/working/eval_zips"))
for zname, out_name in EVAL_DIRS.items():
    zp = pathlib.Path(zips) / zname
    assert zp.exists(), f"missing {zp}"
    with zipfile.ZipFile(zp) as z:
        z.extractall(DATA)
    print("extracted", zname)
# DermNet split -> data/dermnet (train/validation/test) for dermnet_test + class-order root
dzip = snapshot_download("Nirob-jon/cse475-dermnet-split", repo_type="dataset", allow_patterns="DermNetPrepared.zip", local_dir=pathlib.Path("/kaggle/working/eval_zips"))
dnet = DATA / "dermnet"
dnet.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(dzip / "DermNetPrepared.zip") as z:
    z.extractall(dnet)
print("dermnet extracted:", sorted(list(dnet.iterdir())))

In [ ]:
# checkpoint from HF
import huggingface_hub
ckpt_dir = pathlib.Path("/kaggle/working/ckpt")
ckpt_dir.mkdir(parents=True, exist_ok=True)
ckpt = huggingface_hub.hf_hub_download(CKPT_REPO, "phase2_dermnet_best.pt", local_dir=ckpt_dir)
print("checkpoint:", ckpt)

In [ ]:
# light deps only (torch/torchvision preinstalled on Kaggle)
!pip install -q timm tqdm pyyaml imagehash scikit-learn huggingface_hub

In [ ]:
# sanity: splits + class count, and the label_map on disk
import os
for split in ("train", "test"):
    cls_dirs = [d for d in os.listdir(os.path.join(str(dnet), split)) if os.path.isdir(os.path.join(str(dnet), split, d))]
    n = sum(len(os.listdir(os.path.join(str(dnet), split, c))) for c in cls_dirs)
    print(f"dermnet/{split}: {len(cls_dirs)} classes, {n} images")
print("label_map exists:", pathlib.Path("configs/label_map.csv").exists())

In [ ]:
# RUN EVAL (Kaggle profile: preinstalled CUDA torch, batch 32)
!python src/eval_cross.py --checkpoint /kaggle/working/ckpt/phase2_dermnet_best.pt --config configs/phase2_eval_kaggle.yaml
print("---- results file ----")
%cat results/phase2_dermnet_eval.json

## Read the results

The JSON above (also at `results/phase2_dermnet_eval.json`) gives per-set `accuracy` / `macro_f1` / `n`.
Download it from the output panel, or copy it into the paper's results table.